Цель эксперимента: Провести финальное кросс-валидационное сравнение всех 5 обученных моделей (LogReg, RF, LGBM, CatBoost, XGBoost) на тестовой выборке по метрикам ROC-AUC, Accuracy, Precision и Recall, а также построить сравнительный график ROC-кривых.
Какие данные используются: Результаты предсказаний (вероятности и классы) всех ранее сохраненных моделей на основе тестового подмножества cleaned_bnpl_data.csv.
Какие основные выводы: Несмотря на то, что модель XGBoost показала наивысший показатель интегральной метрики $ROC-AUC = 0.8133$, для интеграции в продакшн-сервис была выбрана модель LightGBM ($ROC-AUC = 0.7099$). Данный выбор обусловлен спецификой бизнеса кредитования и BNPL-сервисов:

1. Критическая важность метрики Recall (Полнота): В задачах кредитного скоринга цена ошибки «Ложноотрицательного решения» (пропуск дефолтного клиента, FN) колоссально выше, чем цена «Ложноположительного» (отказ хорошему клиенту, FP). Пропуск неплательщика генерирует прямой убыток в размере всей суммы покупки.
   - LightGBM продемонстрировал абсолютное превосходство по метрике Recall, успешно выявляя 78.36% потенциальных дефолтов.
   - Ближайшие конкуренты (XGBoost и CatBoost), несмотря на высокую метрику Accuracy, показали критически низкий Recall (~21.6% и ~15.5% соответственно), что означает пропуск почти 80% проблемных заемщиков в систему.

2. Эффективность инференса: Помимо бизнес-метрик, LightGBM обеспечивает нативную и быструю работу с категориальными признаками `category` «из коробки» без раздувания признакового пространства через One-Hot Encoding, что критично для обеспечения минимального времени отклика (Latency) нашего REST API.

В этом ноутбуке находится только финальное сравнение обученных моделей на одной и той же тестовой выборке.

- обучение моделей выполняется в отдельных ноутбуках:
  - `01_eda.ipynb` — разведочный анализ данных;
  - `02_model_training.ipynb` — LightGBM;
  - `03_baseline_models.ipynb` — LogisticRegression и RandomForest;
  - `04_boosting_training.ipynb` — XGBoost и CatBoost;
- `05_experiments.ipynb` — только оценка и сравнение метрик на тесте.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import recall_score, precision_score, accuracy_score, roc_auc_score
import joblib
import os

cat_cols = ['Gender', 'Purchase_Category', 'BNPL_Provider', 'Device_Type', 'Connection_Type', 'Browser']

from pathlib import Path
if Path.cwd().name == 'notebooks':
    os.chdir(Path.cwd().parent)
print('Working dir:', Path.cwd())

df = pd.read_csv('data/cleaned_bnpl_data.csv')
if 'Target' not in df.columns:
    if 'Repayment_Status' in df.columns:
        df['Target'] = (df['Repayment_Status'] == 'Defaulted').astype(int)
    else:
        raise KeyError('Target not found in data/cleaned_bnpl_data.csv')
X = df.drop(columns=['Target'])
y = df['Target']

train_idx, test_idx = train_test_split(df.index, test_size=0.20, stratify=y, random_state=42)
X_test = X.loc[test_idx]
y_test = y.loc[test_idx]

X_dummy = pd.get_dummies(X, columns=cat_cols)
X_train_d = X_dummy.loc[train_idx]
X_test_d = X_dummy.loc[test_idx]

scaler = StandardScaler()
scaler.fit(X_train_d)
X_test_scaled = scaler.transform(X_test_d)

X_xgb = X_dummy
X_test_xgb = X_xgb.loc[test_idx]

X_cat = X.copy()
for col in cat_cols:
    X_cat[col] = X_cat[col].astype('category')
X_test_cat = X_cat.loc[test_idx]

X_test_lgb = X_test_cat

artifacts = {
    'LogReg': 'artifacts/models/baseline_logreg.pkl',
    'RF': 'artifacts/models/random_forest.pkl',
    'LGBM': 'artifacts/models/lgbm_risk_model.pkl',
    'XGBoost': 'artifacts/models/xgboost_model.pkl',
    'CatBoost': 'artifacts/models/catboost_model.pkl'
}

loaded_models = {}
for name, path in artifacts.items():
    if os.path.exists(path):
        loaded_models[name] = joblib.load(path)
    else:
        print(f'Warning: model file not found: {path}')

X_map = {
    'LogReg': X_test_scaled,
    'RF': X_test_scaled,
    'LGBM': X_test_lgb,
    'XGBoost': X_test_xgb,
    'CatBoost': X_test_cat
}

results = []
for name, model in loaded_models.items():
    X_input = X_map[name]
    preds = model.predict(X_input)
    probs = model.predict_proba(X_input)[:, 1] if hasattr(model, 'predict_proba') else np.zeros(len(preds))

    results.append({
        'model': name,
        'recall': recall_score(y_test, preds),
        'precision': precision_score(y_test, preds, zero_division=0),
        'accuracy': accuracy_score(y_test, preds),
        'roc_auc': roc_auc_score(y_test, probs) if len(np.unique(y_test)) == 2 else np.nan
    })

if results:
    df_results = pd.DataFrame(results).set_index('model')
    display(df_results)
    os.makedirs('artifacts/experiments', exist_ok=True)
    df_results.to_csv('artifacts/experiments/model_comparison_all.csv')
else:
    print('No models were loaded. Проверьте наличие файлов в artifacts/models.')

Working dir: c:\Users\fedor\Documents\proga\repository-fedi-gr1\project

=== ИТОГОВОЕ СРАВНЕНИЕ МОДЕЛЕЙ ===


,recall,precision,accuracy,roc_auc
model,,,,
LogReg,0.110776,0.704110,0.7829,0.700942
RF,0.162500,0.693015,0.7890,0.713087
LGBM,0.795690,0.353708,0.6153,0.714523
XGBoost,0.216810,0.740795,0.8007,0.813307
CatBoost,0.155603,0.691571,0.7880,0.725711
